## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import mlflow
import mlflow.sklearn
import sys
import os
import pandas as pd
import numpy as np
import time

sys.path.append(os.path.abspath(".."))

from pathlib import Path

PROJECT_ROOT = Path().resolve().parent

# ===================================================== MODELS ========================================================
from src.models import MF_SGD, MF_SVD, content_based_recommendations, user_content_recommendations, train_test_split_per_user, evaluate_hit_rate, evaluate_hit_rate_fast, hybrid_model_recommendations, cold_start_flow, make_hybrid_evaluator

# ==================================================== FEATURES =======================================================
from src.features import merge_movies_with_tags, build_content_matrix, cf_to_numpy, prepare_cf_data

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")

## Data loading

In [4]:
movies = pd.read_csv("../data/processed/movies.csv")
tags =  pd.read_csv("../data/processed/tags.csv")
ratings =  pd.read_csv("../data/processed/ratings.csv")

## Collaborative filtering with matrix factorization

In [5]:
cf_ratings = ratings.copy()
cf_movies = movies.copy()

cf_train_df, cf_test_df, cf_user_map, cf_movie_map = prepare_cf_data(cf_ratings)

(cf_train_users, cf_train_items, cf_train_ratings, cf_test_users, cf_test_items, cf_test_ratings) = cf_to_numpy(cf_train_df, cf_test_df)

- ##### SGD method

In [6]:
target_user_id = 1 

mlflow.set_experiment("Collaborative_Filtering_Methods")

# ==============================================================================
# SGD MATRIX FACTORIZATION RUN
# ==============================================================================
k_sgd = 20
epochs = 3
lr = 0.01
reg = 0.02

with mlflow.start_run(run_name=f"SGD_k{k_sgd}_ep{epochs}"):
    # Log hyperparameters for SGD
    mlflow.log_params({
        "algorithm": "SGD",
        "latent_factors": k_sgd,
        "epochs": epochs,
        "learning_rate": lr,
        "regularization": reg
    })

    sgd_model = MF_SGD(k=k_sgd, epochs=epochs, lr=lr, reg=reg)
    sgd_model.fit(cf_train_users, cf_train_items, cf_train_ratings, len(cf_user_map), len(cf_movie_map))
    
    sgd_preds = sgd_model.predict(cf_test_users, cf_test_items)
    rmse_sgd = np.sqrt(np.mean((cf_test_ratings - sgd_preds) ** 2))
    mlflow.log_metric("rmse", rmse_sgd) 

    mlflow.sklearn.log_model(sgd_model, "sgd_model_artifact")

    print(f"SGD Run Completed. Test RMSE: {rmse_sgd:.4f}")

2026/05/04 17:36:02 INFO mlflow.tracking.fluent: Experiment with name 'Collaborative_Filtering_Methods' does not exist. Creating a new experiment.


[cf-sgd] Epoch 1: RMSE = 0.8896
[cf-sgd] Epoch 2: RMSE = 0.8544
[cf-sgd] Epoch 3: RMSE = 0.8287


2026/05/04 18:44:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 18:44:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


SGD Run Completed. Test RMSE: 0.8261
🏃 View run SGD_k20_ep3 at: http://localhost:5000/#/experiments/1/runs/d5e78bba53484b939844138460e7c891
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [15]:
rec_sgd = sgd_model.recommend(user_id=target_user_id, user_map=cf_user_map, 
                              movie_map=cf_movie_map, df_movies=movies, top_n=10)

print(f"Top 5 recommendations for user {target_user_id}:")
rec_sgd

Top 5 recommendations for user 1:


,title
20251,"Century of the Self, The (2002)"
39767,It's Such a Beautiful Day (2011)
40864,Planet Earth (2006)
42734,Over the Garden Wall (2013)
45982,Band of Brothers (2001)
46131,Planet Earth II (2016)
46366,Cosmos
50005,Blue Planet II (2017)
59010,Twin Peaks (1989)
68338,Twelve Angry Men (1954)


- ##### SVD method

In [8]:
# ==============================================================================
# SVD MATRIX FACTORIZATION RUN
# ==============================================================================
k_svd = 50

with mlflow.start_run(run_name=f"SVD_k{k_svd}"):
    # Log hyperparameters for SVD
    mlflow.log_params({
        "algorithm": "SVD",
        "latent_factors": k_svd
    })

    svd_model = MF_SVD(k=k_svd)
    svd_model.fit(train_df=cf_train_df, n_users=len(cf_user_map), n_movies=len(cf_movie_map))

    svd_preds = svd_model.predict(cf_test_users, cf_test_items)
    rmse_svd = np.sqrt(np.mean((cf_test_ratings - svd_preds)**2))
    mlflow.log_metric("rmse", rmse_svd)
    mlflow.sklearn.log_model(svd_model, "svd_model_artifact")
    
    print(f"SVD Run Completed. Test RMSE: {rmse_svd:.4f}")

2026/05/04 18:48:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 18:48:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


SVD Run Completed. Test RMSE: 0.8862
🏃 View run SVD_k50 at: http://localhost:5000/#/experiments/1/runs/b82336896c544090be5be3ac8cd564c7
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [16]:
rec_svd = svd_model.recommend(user_id=target_user_id, user_map=cf_user_map, 
                              movie_map=cf_movie_map, df_movies=movies, top_n=10)

print(f"Top 5 Recommendations for User {target_user_id}")
rec_svd

Top 5 Recommendations for User 1


,title
256,Star Wars: Episode IV - A New Hope (1977)
534,Blade Runner (1982)
759,Independence Day (a.k.a. ID4) (1996)
834,"Godfather, The (1972)"
897,2001: A Space Odyssey (1968)
1161,Aliens (1986)
1174,Alien (1979)
1181,"Godfather: Part II, The (1974)"
1198,"Terminator, The (1984)"
2742,American Beauty (1999)


## Content Based

### For film

In [6]:
movies_plus_tags = merge_movies_with_tags(movies, tags)
movies_plus_tags.head()

,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,children Disney animation children Disney Disn...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Robin Williams fantasy Robin Williams time tra...
2,3,Grumpier Old Men (1995),Comedy|Romance,comedinha de velhinhos engraÃƒÂ§ada comedinha ...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,characters slurs based on novel or book chick ...
4,5,Father of the Bride Part II (1995),Comedy,Fantasy pregnancy remake family Steve Martin s...


In [5]:
matrix, vectorizer = build_content_matrix(movies_plus_tags)
list_of_recomendation_films_content_based = content_based_recommendations(1, movies_plus_tags, matrix, top_n=10)
list_of_recomendation_films_content_based

,movieId,title
2997,3114,Toy Story 2 (1999)
2243,2355,"Bug's Life, A (1998)"
14761,78499,Toy Story 3 (2010)
4753,4886,"Monsters, Inc. (2001)"
39729,157296,Finding Dory (2016)
6229,6377,Finding Nemo (2003)
8213,8961,"Incredibles, The (2004)"
26515,126405,The Adventures of André and Wally B. (1984)
19789,103141,Monsters University (2013)
18236,95856,Knick Knack (1989)


### For user

In [6]:
target_user_id = 1

list_of_recomendation_films_content_based_user = user_content_recommendations(
    user_id=target_user_id,
    ratings=ratings,
    movies=movies_plus_tags,
    matrix=matrix,
    top_n=10
)
list_of_recomendation_films_content_based_user

,movieId,title
893,920,Gone with the Wind (1939)
1208,1250,"Bridge on the River Kwai, The (1957)"
7148,7303,The Diary of Anne Frank (1959)
1830,1936,Mrs. Miniver (1942)
3355,3475,"Place in the Sun, A (1951)"
1821,1927,All Quiet on the Western Front (1930)
834,858,"Godfather, The (1972)"
1827,1933,"Life of Emile Zola, The (1937)"
1829,1935,How Green Was My Valley (1941)
1251,1293,Gandhi (1982)


In [ ]:
# Split data
ratings_train, ratings_test = train_test_split_per_user(ratings)

# Evaluation
score = evaluate_hit_rate_fast(
    model_func=user_content_recommendations,
    ratings_train=ratings_train,
    ratings_test=ratings_test,
    movies=movies_plus_tags,
    matrix=matrix,
    k=10
)

print(f"Hit Rate@10: {score:.4f}")

  0%|          | 0/200948 [00:00<?, ?it/s]

Hit Rate@10: 0.5323


## Hybrid model

In [7]:
matrix, vectorizer = build_content_matrix(movies_plus_tags)
target_user_id = 1
alpha = 0.5

In [9]:
k_sgd = 20
epochs = 3
lr = 0.01
reg = 0.02

sgd_model = MF_SGD(k=k_sgd, epochs=epochs, lr=lr, reg=reg)
sgd_model.fit(cf_train_users, cf_train_items, cf_train_ratings, len(cf_user_map), len(cf_movie_map))

[cf-sgd] Epoch 1: RMSE = 0.8896
[cf-sgd] Epoch 2: RMSE = 0.8544
[cf-sgd] Epoch 3: RMSE = 0.8287


In [24]:
hybrid_results = hybrid_model_recommendations(
    user_id=target_user_id, 
    ratings=ratings, 
    movies=movies_plus_tags, 
    matrix=matrix, 
    cf_model=sgd_model, 
    user_map=cf_user_map, 
    movie_map=cf_movie_map, 
    alpha=alpha, 
    top_n=10
)

hybrid_results

,movieId,title
1208,1250,"Bridge on the River Kwai, The (1957)"
893,920,Gone with the Wind (1939)
834,858,"Godfather, The (1972)"
3355,3475,"Place in the Sun, A (1951)"
1821,1927,All Quiet on the Western Front (1930)
1829,1935,How Green Was My Valley (1941)
895,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)
1840,1946,Marty (1955)
887,914,My Fair Lady (1964)
1251,1293,Gandhi (1982)


In [ ]:
ratings_train, ratings_test = train_test_split_per_user(ratings)

print(type(sgd_model))

hybrid_eval = make_hybrid_evaluator(
    cf_model=sgd_model,
    user_map=cf_user_map,
    movie_map=cf_movie_map,
    alpha=0.5
)

# Evaluation
score = evaluate_hit_rate_fast(
    model_func=hybrid_eval,
    ratings_train=ratings_train,
    ratings_test=ratings_test,
    movies=movies_plus_tags,
    matrix=matrix,
    k=10
)

print(f"Hit Rate@10: {score:.4f}")

<class 'src.models.MF_SGD'>


100%|██████████| 200948/200948 [6:06:49<00:00,  9.13it/s]    

Hit Rate@10: 0.6002


## Cold start

In [7]:
movies_to_rate = [
    "Toy Story (1995)",
    "Jumanji (1995)",
    "Heat (1995)",
    "Seven (1995)",
    "Braveheart (1995)"
]

ratings_cold_start = {}

for title in movies_to_rate:
    rating_cold_start = float(input(f"Rate '{title}' (1-5): "))
    ratings_cold_start[title] = rating_cold_start

In [ ]:
ratings_path = PROJECT_ROOT / "data" / "processed" / "ratings.csv"

new_user_id = cold_start_flow(
    ratings,
    movies,
    movies_plus_tags,
    matrix,
    content_based_recommendations,
    user_content_recommendations,
    ratings_path=ratings_path
)


    Did you like this movie?

    1 = strongly dislike
    2 = dislike
    3 = neutral
    4 = like
    5 = strongly like
    0 = not seen

    You can also use half ratings (e.g. 1.5, 2.5, 3.5, 4.5)
    
Invalid rating. Try again (0,1,1.5,...,5)

Saved 14 ratings for new user.

--- CONTENT-BASED RECOMMENDATIONS ---

Since you like the movie: Toy Story (1995)
       movieId                                        title
2997      3114                           Toy Story 2 (1999)
2243      2355                         Bug's Life, A (1998)
14761    78499                           Toy Story 3 (2010)
4753      4886                        Monsters, Inc. (2001)
39729   157296                          Finding Dory (2016)
6229      6377                          Finding Nemo (2003)
8213      8961                      Incredibles, The (2004)
26515   126405  The Adventures of André and Wally B. (1984)
19789   103141                   Monsters University (2013)
18236    95856                       

In [ ]:
# Find film
#movies[movies["title"].str.contains("Beautiful Mind", case=False, na=False)]